<a href="https://colab.research.google.com/github/bimaadiwijaya8/-ML_05_Bima-Adiwijaya/blob/main/JS09/JS09-Tugas1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install annoy faiss-cpu hnswlib numpy tabulate -q

In [1]:
# 1. Install dependensi
!pip install annoy faiss-cpu hnswlib numpy -q

import time
import numpy as np
from annoy import AnnoyIndex
import faiss
import hnswlib

# Parameter percobaan
data_sizes = [1000, 1000000]
dimensions = [2, 5]
metrics = ['euclidean', 'cosine']
k = 10

results = []

print("Menjalankan eksperimen benchmark...\n")

for n_data in data_sizes:
    for dim in dimensions:
        for metric in metrics:
            # Generate dataset acak
            X = np.random.random((n_data, dim)).astype(np.float32)
            query = np.random.random((1, dim)).astype(np.float32)

            label_test = f"{n_data} / {dim}D ({metric.capitalize()})"

            # -------------------------------------------------------------
            # A. ANNOY
            # -------------------------------------------------------------
            annoy_metric = 'angular' if metric == 'cosine' else 'euclidean'
            ann_index = AnnoyIndex(dim, annoy_metric)

            start = time.time()
            for i in range(n_data):
                ann_index.add_item(i, X[i])
            ann_index.build(10)
            ann_build = time.time() - start

            start = time.time()
            _ = ann_index.get_nns_by_vector(query[0], k)
            ann_query = time.time() - start

            # -------------------------------------------------------------
            # B. FAISS
            # -------------------------------------------------------------
            if metric == 'euclidean':
                faiss_index = faiss.IndexFlatL2(dim)
                X_faiss = X
                query_faiss = query
            else:
                faiss_index = faiss.IndexFlatIP(dim)
                X_faiss = X / np.linalg.norm(X, axis=1, keepdims=True)
                query_faiss = query / np.linalg.norm(query, axis=1, keepdims=True)

            start = time.time()
            faiss_index.add(X_faiss)
            faiss_build = time.time() - start

            start = time.time()
            _, _ = faiss_index.search(query_faiss, k)
            faiss_query = time.time() - start

            # -------------------------------------------------------------
            # C. HNSW
            # -------------------------------------------------------------
            hnsw_space = 'cosine' if metric == 'cosine' else 'l2'
            hnsw_index = hnswlib.Index(space=hnsw_space, dim=dim)

            start = time.time()
            hnsw_index.init_index(max_elements=n_data, ef_construction=200, M=16)
            hnsw_index.add_items(X)
            hnsw_build = time.time() - start

            hnsw_index.set_ef(50)
            start = time.time()
            _, _ = hnsw_index.knn_query(query, k=k)
            hnsw_query = time.time() - start

            # Format teks bertumpuk di dalam 1 sel (menggunakan <br>)
            results.append({
                "label": label_test,
                "annoy": f"Build: {ann_build:.4f}s<br>Query: {ann_query:.5f}s",
                "faiss": f"Build: {faiss_build:.4f}s<br>Query: {faiss_query:.5f}s",
                "hnsw": f"Build: {hnsw_build:.4f}s<br>Query: {hnsw_query:.5f}s"
            })

# -------------------------------------------------------------
# Generate Teks Tabel Markdown
# -------------------------------------------------------------
md_table = "| Jumlah Data/Dimensi | ANNOY | FAISS | HNSW |\n"
md_table += "| :--- | :--- | :--- | :--- |\n"

for r in results:
    md_table += f"| **{r['label']}** | {r['annoy']} | {r['faiss']} | {r['hnsw']} |\n"

print("="*60)
print("HASIL TEKS MARKDOWN (COPY SEMUA TEKS DI BAWAH INI):")
print("="*60 + "\n")
print(md_table)

Menjalankan eksperimen benchmark...

HASIL TEKS MARKDOWN (COPY SEMUA TEKS DI BAWAH INI):

| Jumlah Data/Dimensi | ANNOY | FAISS | HNSW |
| :--- | :--- | :--- | :--- |
| **1000 / 2D (Euclidean)** | Build: 0.0217s<br>Query: 0.00007s | Build: 0.0001s<br>Query: 0.00007s | Build: 0.0436s<br>Query: 0.00006s |
| **1000 / 2D (Cosine)** | Build: 0.0279s<br>Query: 0.00008s | Build: 0.0000s<br>Query: 0.00004s | Build: 0.0343s<br>Query: 0.00007s |
| **1000 / 5D (Euclidean)** | Build: 0.0181s<br>Query: 0.00006s | Build: 0.0000s<br>Query: 0.00005s | Build: 0.0613s<br>Query: 0.00008s |
| **1000 / 5D (Cosine)** | Build: 0.0229s<br>Query: 0.00006s | Build: 0.0000s<br>Query: 0.00004s | Build: 0.0630s<br>Query: 0.00006s |
| **1000000 / 2D (Euclidean)** | Build: 13.8803s<br>Query: 0.00009s | Build: 0.0027s<br>Query: 0.00580s | Build: 69.5183s<br>Query: 0.00006s |
| **1000000 / 2D (Cosine)** | Build: 40.6544s<br>Query: 0.00009s | Build: 0.0009s<br>Query: 0.00143s | Build: 98.1267s<br>Query: 0.00010s |
| **

HASIL TEKS MARKDOWN (COPY SEMUA TEKS DI BAWAH INI):
============================================================

| Jumlah Data/Dimensi | ANNOY | FAISS | HNSW |
| :--- | :--- | :--- | :--- |
| **1000 / 2D (Euclidean)** | Build: 0.0217s<br>Query: 0.00007s | Build: 0.0001s<br>Query: 0.00007s | Build: 0.0436s<br>Query: 0.00006s |
| **1000 / 2D (Cosine)** | Build: 0.0279s<br>Query: 0.00008s | Build: 0.0000s<br>Query: 0.00004s | Build: 0.0343s<br>Query: 0.00007s |
| **1000 / 5D (Euclidean)** | Build: 0.0181s<br>Query: 0.00006s | Build: 0.0000s<br>Query: 0.00005s | Build: 0.0613s<br>Query: 0.00008s |
| **1000 / 5D (Cosine)** | Build: 0.0229s<br>Query: 0.00006s | Build: 0.0000s<br>Query: 0.00004s | Build: 0.0630s<br>Query: 0.00006s |
| **1000000 / 2D (Euclidean)** | Build: 13.8803s<br>Query: 0.00009s | Build: 0.0027s<br>Query: 0.00580s | Build: 69.5183s<br>Query: 0.00006s |
| **1000000 / 2D (Cosine)** | Build: 40.6544s<br>Query: 0.00009s | Build: 0.0009s<br>Query: 0.00143s | Build: 98.1267s<br>Query: 0.00010s |
| **1000000 / 5D (Euclidean)** | Build: 11.1261s<br>Query: 0.00009s | Build: 0.0064s<br>Query: 0.00604s | Build: 113.9855s<br>Query: 0.00008s |
| **1000000 / 5D (Cosine)** | Build: 14.1904s<br>Query: 0.00010s | Build: 0.0049s<br>Query: 0.00230s | Build: 102.6484s<br>Query: 0.00009s |